In [1]:
import json
import random
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

random.seed(42)

In [2]:
data_dir = Path("../data")

csv_path = data_dir / "scanner_style_llm_dataset.csv"
jsonl_full_path = data_dir / "qwen7b_scanner_dataset.jsonl"
jsonl_train_path = data_dir / "qwen7b_scanner_train.jsonl"
jsonl_val_path = data_dir / "qwen7b_scanner_val.jsonl"

data_dir.mkdir(exist_ok=True)

In [3]:
licenses = {
    "Low": [
        ("MIT", "Permissive"),
        ("Apache-2.0", "Permissive"),
        ("BSD-3-Clause", "Permissive"),
        ("BSD-2-Clause", "Permissive"),
        ("CC0-1.0", "Public Domain"),
        ("Unlicense", "Public Domain"),
    ],
    "Medium": [
        ("LGPL-2.1", "Weak Copyleft"),
        ("LGPL-3.0", "Weak Copyleft"),
        ("MPL-2.0", "Weak Copyleft"),
        ("EPL-2.0", "Weak Copyleft"),
        ("CDDL-1.0", "Weak Copyleft"),
    ],
    "High": [
        ("GPL-2.0", "Strong Copyleft"),
        ("GPL-3.0", "Strong Copyleft"),
        ("AGPL-3.0", "Strong Copyleft"),
        ("Proprietary", "Restricted"),
        ("Unknown", "Unknown"),
        ("Custom License", "Unknown"),
    ],
}

ecosystems = [
    ("python", "pip"),
    ("node", "npm"),
    ("java", "maven"),
    ("rust", "cargo"),
    ("go", "go-mod"),
]

project_types = [
    "commercial SaaS platform",
    "commercial closed-source software",
    "enterprise analytics platform",
    "mobile application",
    "desktop application",
    "embedded firmware",
    "AI inference platform",
    "cloud CRM platform",
    "banking middleware",
    "healthcare portal",
    "DevOps automation platform",
    "robotics control system",
    "industrial IoT platform",
    "online education platform",
    "financial reporting system",
    "cybersecurity monitoring tool",
]

usages = [
    "core library",
    "backend framework",
    "shared library",
    "plugin framework",
    "SDK integration",
    "API service",
    "utility package",
    "frontend component",
    "workflow engine",
    "data processing module",
    "model serving framework",
    "automation library",
    "embedded library",
    "messaging framework",
    "visualization library",
]

dependency_scopes = ["runtime", "build", "test", "optional"]
direct_types = ["direct", "transitive"]
dev_dependency_values = ["yes", "no"]
linking_types = ["dynamic", "static", "service", "unknown"]

In [4]:
def build_reason(row):
    risk = row["risk"]
    license_name = row["license"]
    license_family = row["license_family"]

    if risk == "Low":
        return (
            f"{license_name} is low risk because obligations are satisfied "
            f"and required notices or license text are preserved."
        )

    if risk == "Medium":
        return (
            f"{license_name} creates moderate compliance risk because weak copyleft "
            f"or contextual obligations require review."
        )

    return (
        f"{license_name} is high risk because strong copyleft, unknown, custom, "
        f"or restricted terms may create significant compliance obligations."
    )

In [5]:
def generate_context(risk):
    if risk == "Low":
        return {
            "network_exposed": random.choice(["no", "no", "yes"]),
            "commercial_use": "yes",
            "attribution_notice": "preserved",
            "license_text": "included",
            "source_modified": random.choice(["no", "no", "possible"]),
            "redistribution": random.choice(["no", "no", "possible"]),
            "license_confidence": "high",
        }

    if risk == "Medium":
        return {
            "network_exposed": random.choice(["no", "possible", "yes"]),
            "commercial_use": "yes",
            "attribution_notice": random.choice(["preserved", "preserved", "unknown"]),
            "license_text": random.choice(["included", "included", "unknown"]),
            "source_modified": random.choice(["possible", "yes", "no"]),
            "redistribution": random.choice(["possible", "yes", "no"]),
            "license_confidence": random.choice(["medium", "high"]),
        }

    return {
        "network_exposed": random.choice(["yes", "unknown", "possible"]),
        "commercial_use": "yes",
        "attribution_notice": random.choice(["unknown", "missing", "preserved"]),
        "license_text": random.choice(["unknown", "missing", "included"]),
        "source_modified": random.choice(["possible", "yes", "unknown"]),
        "redistribution": random.choice(["yes", "unknown", "possible"]),
        "license_confidence": random.choice(["low", "medium"]),
    }

In [6]:
rows = []

target_per_class = 400

for risk in ["Low", "Medium", "High"]:
    for i in range(target_per_class):
        license_name, license_family = random.choice(licenses[risk])
        ecosystem, package_manager = random.choice(ecosystems)

        row = {
            "package": f"{ecosystem}-package-{risk.lower()}-{i}",
            "version": f"{random.randint(0, 5)}.{random.randint(0, 20)}.{random.randint(0, 30)}",
            "ecosystem": ecosystem,
            "package_manager": package_manager,
            "license": license_name,
            "license_family": license_family,
            "dependency_scope": random.choice(dependency_scopes),
            "direct_or_transitive": random.choice(direct_types),
            "is_dev_dependency": random.choice(dev_dependency_values),
            "project_type": random.choice(project_types),
            "distribution_model": random.choice(["hosted", "distributed", "internal-only"]),
            "usage": random.choice(usages),
            "linking_type": random.choice(linking_types),
            **generate_context(risk),
            "risk": risk,
        }

        row["reason"] = build_reason(row)
        rows.append(row)

df = pd.DataFrame(rows)

print(df.shape)
print(df["risk"].value_counts())

df.head()

(1200, 22)
risk
Low       400
Medium    400
High      400
Name: count, dtype: int64


,package,version,ecosystem,package_manager,license,license_family,dependency_scope,direct_or_transitive,is_dev_dependency,project_type,...,linking_type,network_exposed,commercial_use,attribution_notice,license_text,source_modified,redistribution,license_confidence,risk,reason
0,python-package-low-0,0.8.7,python,pip,Unlicense,Public Domain,build,direct,yes,enterprise analytics platform,...,dynamic,no,yes,preserved,included,no,no,high,Low,Unlicense is low risk because obligations are ...
1,go-package-low-1,4.0.17,go,go-mod,Apache-2.0,Permissive,build,transitive,yes,financial reporting system,...,dynamic,no,yes,preserved,included,possible,no,high,Low,Apache-2.0 is low risk because obligations are...
2,java-package-low-2,1.6.30,java,maven,BSD-3-Clause,Permissive,test,direct,yes,industrial IoT platform,...,service,yes,yes,preserved,included,no,no,high,Low,BSD-3-Clause is low risk because obligations a...
3,rust-package-low-3,4.3.29,rust,cargo,Unlicense,Public Domain,optional,direct,no,robotics control system,...,dynamic,no,yes,preserved,included,possible,no,high,Low,Unlicense is low risk because obligations are ...
4,python-package-low-4,1.3.12,python,pip,BSD-3-Clause,Permissive,test,transitive,no,embedded firmware,...,static,yes,yes,preserved,included,no,possible,high,Low,BSD-3-Clause is low risk because obligations a...


In [7]:
df.to_csv(csv_path, index=False)

print("Saved CSV:", csv_path)

Saved CSV: ../data/scanner_style_llm_dataset.csv


In [8]:
def build_user_prompt(row):
    return f"""
You are an OSS compliance risk analysis assistant.

Classify this scanner output.

Package: {row["package"]}
Version: {row["version"]}
Ecosystem: {row["ecosystem"]}
Package Manager: {row["package_manager"]}
License: {row["license"]}
License Family: {row["license_family"]}
Dependency Scope: {row["dependency_scope"]}
Direct or Transitive: {row["direct_or_transitive"]}
Development Dependency: {row["is_dev_dependency"]}
Project Type: {row["project_type"]}
Distribution Model: {row["distribution_model"]}
Usage: {row["usage"]}
Linking Type: {row["linking_type"]}
Network Exposed: {row["network_exposed"]}
Commercial Use: {row["commercial_use"]}
Attribution Notice: {row["attribution_notice"]}
License Text: {row["license_text"]}
Source Modified: {row["source_modified"]}
Redistribution: {row["redistribution"]}
License Confidence: {row["license_confidence"]}

Return exactly in this format:
Risk: <Low|Medium|High>
Reason: <one short sentence>
""".strip()


def build_assistant_response(row):
    return f"""Risk: {row["risk"]}
Reason: {row["reason"]}"""

In [9]:
records = []

for _, row in df.iterrows():
    records.append({
        "messages": [
            {
                "role": "user",
                "content": build_user_prompt(row)
            },
            {
                "role": "assistant",
                "content": build_assistant_response(row)
            }
        ]
    })

print("Total records:", len(records))
records[0]

Total records: 1200


{'messages': [{'role': 'user',
   'content': 'You are an OSS compliance risk analysis assistant.\n\nClassify this scanner output.\n\nPackage: python-package-low-0\nVersion: 0.8.7\nEcosystem: python\nPackage Manager: pip\nLicense: Unlicense\nLicense Family: Public Domain\nDependency Scope: build\nDirect or Transitive: direct\nDevelopment Dependency: yes\nProject Type: enterprise analytics platform\nDistribution Model: internal-only\nUsage: utility package\nLinking Type: dynamic\nNetwork Exposed: no\nCommercial Use: yes\nAttribution Notice: preserved\nLicense Text: included\nSource Modified: no\nRedistribution: no\nLicense Confidence: high\n\nReturn exactly in this format:\nRisk: <Low|Medium|High>\nReason: <one short sentence>'},
  {'role': 'assistant',
   'content': 'Risk: Low\nReason: Unlicense is low risk because obligations are satisfied and required notices or license text are preserved.'}]}

In [10]:
with open(jsonl_full_path, "w", encoding="utf-8") as f:
    for record in records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("Saved:", jsonl_full_path)

Saved: ../data/qwen7b_scanner_dataset.jsonl


In [11]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["risk"]
)

print("Train:", train_df.shape)
print(train_df["risk"].value_counts())

print("\nValidation:", val_df.shape)
print(val_df["risk"].value_counts())

Train: (960, 22)
risk
Low       320
Medium    320
High      320
Name: count, dtype: int64

Validation: (240, 22)
risk
High      80
Medium    80
Low       80
Name: count, dtype: int64


In [12]:
def save_jsonl(dataframe, path):
    records = []

    for _, row in dataframe.iterrows():
        records.append({
            "messages": [
                {
                    "role": "user",
                    "content": build_user_prompt(row)
                },
                {
                    "role": "assistant",
                    "content": build_assistant_response(row)
                }
            ]
        })

    with open(path, "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    print(f"Saved {len(records)} records to {path}")


save_jsonl(train_df, jsonl_train_path)
save_jsonl(val_df, jsonl_val_path)

Saved 960 records to ../data/qwen7b_scanner_train.jsonl
Saved 240 records to ../data/qwen7b_scanner_val.jsonl


In [13]:
with open(jsonl_train_path, "r", encoding="utf-8") as f:
    sample = json.loads(f.readline())

sample

{'messages': [{'role': 'user',
   'content': 'You are an OSS compliance risk analysis assistant.\n\nClassify this scanner output.\n\nPackage: node-package-low-225\nVersion: 4.19.19\nEcosystem: node\nPackage Manager: npm\nLicense: MIT\nLicense Family: Permissive\nDependency Scope: test\nDirect or Transitive: direct\nDevelopment Dependency: no\nProject Type: AI inference platform\nDistribution Model: internal-only\nUsage: data processing module\nLinking Type: static\nNetwork Exposed: yes\nCommercial Use: yes\nAttribution Notice: preserved\nLicense Text: included\nSource Modified: no\nRedistribution: no\nLicense Confidence: high\n\nReturn exactly in this format:\nRisk: <Low|Medium|High>\nReason: <one short sentence>'},
  {'role': 'assistant',
   'content': 'Risk: Low\nReason: MIT is low risk because obligations are satisfied and required notices or license text are preserved.'}]}